# Lesson 4A: Gaussian Mixture Models - Theory

## The Soft Clustering Problem 🎨

You're a data scientist at Spotify analyzing **user listening patterns** to create personalized playlists. You have features like:
- Average tempo (BPM)
- Energy level
- Danceability
- Valence (happiness)

**K-Means clustering** groups users into hard categories: "Rock fans" OR "Pop fans" OR "Classical fans". But real users don't fit neat boxes:
- Some users are 70% Rock, 30% Pop
- Others are 50% Classical, 40% Jazz, 10% Electronic
- Musical taste is a **mixture**, not a category

**You need:**
1. ✅ **Soft assignments** - Users can belong to multiple clusters with probabilities
2. ✅ **Uncertainty quantification** - How confident is each assignment?
3. ✅ **Generative model** - Can sample new "typical" users from each cluster
4. ✅ **Probabilistic foundation** - Mathematical rigor for decision-making

**Enter Gaussian Mixture Models (GMM)** - the probabilistic cousin of K-Means!

## Real-World Applications

🎵 **Music/Content Recommendation**
- User taste modeling (Spotify, Pandora)
- Content-based filtering with uncertainty
- Hybrid recommendations

🏥 **Medical Diagnosis**
- Disease subtypes with overlapping symptoms
- Patient risk stratification
- Biomarker discovery with uncertainty

📊 **Financial Modeling**
- Market regime detection (bull/bear/sideways)
- Portfolio risk assessment
- Credit scoring with probability

🖼️ **Computer Vision**
- Image segmentation with soft boundaries
- Background/foreground separation
- Color quantization

🔬 **Bioinformatics**
- Gene expression clustering
- Cell type identification
- Protein structure prediction

## What You'll Learn

1. ✅ **Mixture of Gaussians** - Mathematical formulation
2. ✅ **EM Algorithm** - Expectation-Maximization derivation
3. ✅ **From-scratch implementation** - Complete working code
4. ✅ **Model selection** - BIC and AIC for choosing K
5. ✅ **Comparison with K-Means** - When to use each

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import multivariate_normal
from sklearn.datasets import make_blobs

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("✅ Libraries loaded!")

## Gaussian Mixture Model: Mathematical Foundation

### The Generative Story

**GMM assumes data is generated by a mixture of K Gaussian distributions:**

1. **Choose a cluster** k with probability π_k (mixing coefficient)
   - π_k ≥ 0
   - Σ π_k = 1

2. **Sample from that cluster's Gaussian** with parameters (μ_k, Σ_k)
   - μ_k = mean vector
   - Σ_k = covariance matrix

### Probability Density Function

For a data point **x**, the probability density is:

$$p(x) = \sum_{k=1}^{K} \pi_k \mathcal{N}(x | \mu_k, \Sigma_k)$$

Where:
- K = number of mixture components (clusters)
- π_k = mixing coefficient (prior probability of cluster k)
- 𝒩(x | μ_k, Σ_k) = Multivariate Gaussian PDF

$$\mathcal{N}(x | \mu, \Sigma) = \frac{1}{(2\pi)^{D/2} |\Sigma|^{1/2}} \exp\left(-\frac{1}{2}(x-\mu)^T \Sigma^{-1} (x-\mu)\right)$$

### Responsibilities (Soft Assignments)

**Posterior probability** that point x belongs to cluster k:

$$\gamma_k(x) = \frac{\pi_k \mathcal{N}(x | \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \pi_j \mathcal{N}(x | \mu_j, \Sigma_j)}$$

This is **Bayes' theorem**:

$$P(z=k | x) = \frac{P(x | z=k) P(z=k)}{P(x)}$$

**Responsibilities sum to 1** for each point:
$$\sum_{k=1}^{K} \gamma_k(x) = 1$$

### Connection to K-Means

**K-Means is a special case of GMM:**
- Hard assignments: γ_k(x) ∈ {0, 1}
- Spherical covariances: Σ_k = σ²I (all clusters same size)
- Equal mixing: π_k = 1/K

**GMM generalizes K-Means:**
- Soft assignments: γ_k(x) ∈ [0, 1]
- Arbitrary covariances: Σ_k can be full, diagonal, spherical, or tied
- Learned mixing: π_k learned from data

## The EM Algorithm

**Problem**: We can't directly maximize the likelihood because we don't know which cluster generated each point.

**Solution**: Expectation-Maximization (EM) algorithm

### Algorithm Overview

**Repeat until convergence:**

1. **E-step (Expectation)**
   - Compute responsibilities γ_k(x_n) for all points
   - "If I knew the parameters, what are the cluster assignments?"

2. **M-step (Maximization)**
   - Update parameters (π_k, μ_k, Σ_k) using responsibilities
   - "If I knew the assignments, what are the best parameters?"

### E-Step: Compute Responsibilities

For each point n and cluster k:

$$\gamma_{nk} = \frac{\pi_k \mathcal{N}(x_n | \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \pi_j \mathcal{N}(x_n | \mu_j, \Sigma_j)}$$

### M-Step: Update Parameters

**Effective number of points in cluster k:**

$$N_k = \sum_{n=1}^{N} \gamma_{nk}$$

**Update mixing coefficients:**

$$\pi_k = \frac{N_k}{N}$$

**Update means:**

$$\mu_k = \frac{1}{N_k} \sum_{n=1}^{N} \gamma_{nk} x_n$$

**Update covariances:**

$$\Sigma_k = \frac{1}{N_k} \sum_{n=1}^{N} \gamma_{nk} (x_n - \mu_k)(x_n - \mu_k)^T$$

### Why EM Works

**EM is guaranteed to:**
- Increase the log-likelihood at each iteration (or stay the same)
- Converge to a local maximum

**But:**
- May converge to different local maxima depending on initialization
- Use multiple random restarts (like K-Means++)

In [ ]:
# Visualize Gaussian distributions
from matplotlib.patches import Ellipse

# Create 2D Gaussians with different properties
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Gaussian 1: Spherical (circular)
ax = axes[0]
mu1 = [0, 0]
cov1 = [[1, 0], [0, 1]]
x, y = np.mgrid[-4:4:.01, -4:4:.01]
pos = np.dstack((x, y))
rv1 = multivariate_normal(mu1, cov1)
ax.contour(x, y, rv1.pdf(pos), levels=10, cmap='Blues')
ax.set_title('Spherical Gaussian\nΣ = I (equal variance, no correlation)', fontweight='bold')
ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

# Gaussian 2: Diagonal (elliptical, axis-aligned)
ax = axes[1]
mu2 = [0, 0]
cov2 = [[2, 0], [0, 0.5]]
rv2 = multivariate_normal(mu2, cov2)
ax.contour(x, y, rv2.pdf(pos), levels=10, cmap='Greens')
ax.set_title('Diagonal Gaussian\n(different variances, no correlation)', fontweight='bold')
ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

# Gaussian 3: Full (elliptical, rotated)
ax = axes[2]
mu3 = [0, 0]
cov3 = [[2, 1.5], [1.5, 2]]
rv3 = multivariate_normal(mu3, cov3)
ax.contour(x, y, rv3.pdf(pos), levels=10, cmap='Reds')
ax.set_title('Full Covariance Gaussian\n(correlation between features)', fontweight='bold')
ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.suptitle('Three Types of Gaussian Covariance Structures', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

print("✅ Gaussian types visualized!")
print("\nCovariance types:")
print("• Spherical: All features same variance, uncorrelated")
print("• Diagonal: Different variances, uncorrelated")
print("• Full: Different variances, can be correlated")

## GMM Implementation from Scratch

Let's build a complete GMM with the EM algorithm!

In [ ]:
class GaussianMixtureModel:
    """
    Gaussian Mixture Model with EM algorithm.
    
    Supports spherical, diagonal, and full covariance types.
    """
    
    def __init__(self, n_components=3, covariance_type='full', max_iter=100, tol=1e-4, random_state=42):
        """
        Parameters:
            n_components: Number of mixture components (clusters)
            covariance_type: 'spherical', 'diagonal', or 'full'
            max_iter: Maximum EM iterations
            tol: Convergence tolerance for log-likelihood
            random_state: Random seed
        """
        self.n_components = n_components
        self.covariance_type = covariance_type
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        
        # Parameters (learned by EM)
        self.weights_ = None  # π_k (mixing coefficients)
        self.means_ = None     # μ_k
        self.covariances_ = None  # Σ_k
        self.converged_ = False
        self.n_iter_ = 0
        
    def _initialize_parameters(self, X):
        """Initialize parameters using K-Means++."""
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        
        # Initialize means with K-Means++ strategy
        means = []
        first_idx = np.random.randint(n_samples)
        means.append(X[first_idx])
        
        for _ in range(1, self.n_components):
            distances = np.min([np.sum((X - mu)**2, axis=1) for mu in means], axis=0)
            probs = distances / distances.sum()
            next_idx = np.random.choice(n_samples, p=probs)
            means.append(X[next_idx])
        
        self.means_ = np.array(means)
        
        # Initialize covariances
        if self.covariance_type == 'spherical':
            self.covariances_ = np.ones(self.n_components)
        elif self.covariance_type == 'diagonal':
            self.covariances_ = np.ones((self.n_components, n_features))
        else:  # full
            self.covariances_ = np.array([np.eye(n_features) for _ in range(self.n_components)])
        
        # Initialize weights (uniform)
        self.weights_ = np.ones(self.n_components) / self.n_components
    
    def _gaussian_pdf(self, X, mean, cov):
        """Compute Gaussian PDF."""
        n_features = X.shape[1]
        
        if self.covariance_type == 'spherical':
            cov_matrix = cov * np.eye(n_features)
        elif self.covariance_type == 'diagonal':
            cov_matrix = np.diag(cov)
        else:  # full
            cov_matrix = cov
        
        # Add regularization for numerical stability
        cov_matrix = cov_matrix + 1e-6 * np.eye(n_features)
        
        return multivariate_normal(mean, cov_matrix).pdf(X)
    
    def _e_step(self, X):
        """E-step: Compute responsibilities."""
        n_samples = X.shape[0]
        responsibilities = np.zeros((n_samples, self.n_components))
        
        # Compute weighted probabilities
        for k in range(self.n_components):
            responsibilities[:, k] = self.weights_[k] * self._gaussian_pdf(X, self.means_[k], self.covariances_[k])
        
        # Normalize to get responsibilities (avoid division by zero)
        resp_sum = responsibilities.sum(axis=1, keepdims=True)
        resp_sum[resp_sum == 0] = 1e-10
        responsibilities /= resp_sum
        
        return responsibilities
    
    def _m_step(self, X, responsibilities):
        """M-step: Update parameters."""
        n_samples, n_features = X.shape
        
        # Effective number of points per component
        Nk = responsibilities.sum(axis=0)
        
        # Update weights
        self.weights_ = Nk / n_samples
        
        # Update means
        self.means_ = (responsibilities.T @ X) / Nk[:, np.newaxis]
        
        # Update covariances
        for k in range(self.n_components):
            diff = X - self.means_[k]
            
            if self.covariance_type == 'spherical':
                self.covariances_[k] = np.sum(responsibilities[:, k, np.newaxis] * diff**2) / (Nk[k] * n_features)
            elif self.covariance_type == 'diagonal':
                self.covariances_[k] = np.sum(responsibilities[:, k, np.newaxis] * diff**2, axis=0) / Nk[k]
            else:  # full
                weighted_diff = responsibilities[:, k, np.newaxis] * diff
                self.covariances_[k] = (weighted_diff.T @ diff) / Nk[k]
    
    def _compute_log_likelihood(self, X):
        """Compute log-likelihood."""
        n_samples = X.shape[0]
        log_likelihood = 0
        
        for k in range(self.n_components):
            log_likelihood += self.weights_[k] * self._gaussian_pdf(X, self.means_[k], self.covariances_[k])
        
        return np.sum(np.log(log_likelihood + 1e-10))
    
    def fit(self, X):
        """Fit GMM using EM algorithm."""
        self._initialize_parameters(X)
        
        prev_log_likelihood = -np.inf
        
        for iteration in range(self.max_iter):
            # E-step
            responsibilities = self._e_step(X)
            
            # M-step
            self._m_step(X, responsibilities)
            
            # Check convergence
            log_likelihood = self._compute_log_likelihood(X)
            
            if abs(log_likelihood - prev_log_likelihood) < self.tol:
                self.converged_ = True
                self.n_iter_ = iteration + 1
                break
            
            prev_log_likelihood = log_likelihood
            
            if (iteration + 1) % 10 == 0:
                print(f"  Iteration {iteration+1}/{self.max_iter}, Log-likelihood: {log_likelihood:.2f}")
        
        if not self.converged_:
            self.n_iter_ = self.max_iter
            print(f"  EM did not converge after {self.max_iter} iterations")
        
        return self
    
    def predict_proba(self, X):
        """Predict cluster probabilities (responsibilities)."""
        return self._e_step(X)
    
    def predict(self, X):
        """Predict cluster labels (hard assignment)."""
        return self.predict_proba(X).argmax(axis=1)

print("✅ GMM class implemented from scratch!")
print("   Methods: fit(), predict(), predict_proba()")
print("   Supports: spherical, diagonal, full covariances")

In [ ]:
# Test GMM on synthetic data
print("Testing GMM on overlapping clusters...")
print("="*60)

# Create overlapping clusters
X, y_true = make_blobs(n_samples=300, centers=3, n_features=2,
                       cluster_std=1.0, random_state=42)

# Fit GMM
print("\nFitting GMM with EM algorithm...")
gmm = GaussianMixtureModel(n_components=3, covariance_type='full', max_iter=50)
gmm.fit(X)

print(f"\n✅ Converged: {gmm.converged_}")
print(f"   Iterations: {gmm.n_iter_}")
print(f"   Mixing weights: {gmm.weights_}")

# Get predictions
y_pred = gmm.predict(X)
y_proba = gmm.predict_proba(X)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Ground truth
ax = axes[0]
scatter = ax.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', s=50, alpha=0.6, edgecolors='k')
ax.set_title('Ground Truth', fontweight='bold', fontsize=14)
ax.set_xlabel('Feature 1', fontweight='bold')
ax.set_ylabel('Feature 2', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax)

# Hard assignment (like K-Means)
ax = axes[1]
scatter = ax.scatter(X[:, 0], X[:, 1], c=y_pred, cmap='viridis', s=50, alpha=0.6, edgecolors='k')
# Plot learned means
ax.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='red', s=300, marker='*',
          edgecolors='black', linewidths=2, label='Learned means', zorder=10)
ax.set_title('GMM Hard Assignment', fontweight='bold', fontsize=14)
ax.set_xlabel('Feature 1', fontweight='bold')
ax.set_ylabel('Feature 2', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax)

# Soft assignment (uncertainty visualization)
ax = axes[2]
# Color by most probable cluster, size by confidence
max_proba = y_proba.max(axis=1)
scatter = ax.scatter(X[:, 0], X[:, 1], c=y_pred, s=max_proba*200, 
                    cmap='viridis', alpha=0.6, edgecolors='k')
ax.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='red', s=300, marker='*',
          edgecolors='black', linewidths=2, zorder=10)
ax.set_title('GMM Soft Assignment\n(size = confidence)', fontweight='bold', fontsize=14)
ax.set_xlabel('Feature 1', fontweight='bold')
ax.set_ylabel('Feature 2', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='Cluster')

plt.suptitle('GMM Results: Hard vs Soft Assignments', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

print("\n✅ GMM working! Notice soft assignments show uncertainty.")

## Summary

🎯 **What We Learned:**

1. **GMM** = probabilistic, soft clustering via mixture of Gaussians
2. **EM algorithm** = iterative E-step (responsibilities) + M-step (parameters)
3. **Soft assignments** = points can belong to multiple clusters with probabilities
4. **Covariance types** = spherical, diagonal, full for different cluster shapes
5. **Connection to K-Means** = GMM generalizes K-Means with soft assignments

🔬 **Key Advantages:**

- ✅ **Soft clustering** with uncertainty quantification
- ✅ **Generative model** can sample new data
- ✅ **Flexible shapes** via covariance matrices
- ✅ **Probabilistic foundation** for decision-making

🚀 **Next Steps:**

In **Lesson 4B**, we'll:
- Use scikit-learn's production GMM
- Model selection with BIC/AIC
- Compare covariance types
- Apply to real clustering problems

---

**Key Takeaway**: GMM provides **soft clustering with uncertainty**, perfect when data points naturally belong to multiple categories or when you need probabilistic assignments!